# 🎬 TMDB Interactive Movie EDA

**Author:** Vinícius Mangueira
**Date:** May 13, 2025
**Context:**
This notebook delivers a comprehensive, interactive Exploratory Data Analysis of popular movies using the TMDB API, combining data engineering best practices, rigorous analysis, and advanced visualizations.

---

## Key Highlights & Objectives

* **🔒 Best Practices:** Project structure, versioning, and reproducibility guidelines.
* **🔧 Engineering Rigor:** Code organization, modular functions, and testing considerations.
* **🛠 Data Engineering Excellence:** Efficient data acquisition, raw JSON persistence, and automated pipelines.
* **📊 Holistic EDA:** Static and interactive charts to explore ratings, release trends, and genre breakdowns.
* **🔄 Dynamic Interactivity:** Widgets and Plotly controls for on-the-fly data slicing and insights.
* **📈 Metrics & Modeling:** Computation of key performance metrics, financial enrichments, and a basic predictive model for movie ratings.

---

## Notebook Structure & Cell Overview

1. **Introduction & Best Practices** – Project goals, setup conventions, and reproducibility checklist.
2. **Engineering Rigor** – Code modularization and testing strategy notes.
3. **Data Engineering Excellence** – TMDB API configuration, data-fetching loops, and raw JSON storage.
4. **Interactive & Holistic Exploratory Data Analysis**
   4.1 ⭐ Rating Distribution
   4.2 ⏳ Release Trends by Year
   4.3 🎯 Genre Popularity
5. **🔄 Interactive Popularity vs. Rating Scatter** – Custom widgets for dynamic filtering.
6. **Compute & Visualize Key Metrics** – Aggregate statistics and summary tables.
7. **🚀 Enrich Dataset with Financials** – Merging box office and budget data.
8. **🎯 Summary Metrics by Genre** – Comparative genre-level dashboards.
9. **🔢 Predictive Modeling: Linear Regression** – Training and evaluation of a simple rating predictor.

> 💡 **Tip:** Use the sidebar table of contents to navigate directly to any section and adjust widget parameters to explore different scenarios.


## 1. 📦 Libraries Import and Environment Setup

This section prepares the working environment and ensures we have everything needed to access the TMDB API:

1. **Dependencies Import**  
   - Standard library modules: `os`, `json`, `time`, and `pathlib.Path`.  
   - HTTP requests with `requests`.  
   - Environment variable management with `dotenv`.  
   - Data handling and visualization: `pandas`, `plotly.express`, `plotly.io`, and `ipywidgets`.  
   - Inline display in Jupyter via `IPython.display.display`.

2. **Loading Environment Variables**  
   - Calls `load_dotenv()` to import variables from the `.env` file.  
   - Reads the `TMDB_BEARER` authentication token from the environment.

3. **Token Validation**  
   - Uses `assert TMDB_BEARER` to halt execution if the token is missing, showing the error message:  

In [9]:
#Import the necessary libraries
# -*- coding: utf-8 -*-
import os
import json
from pathlib import Path
import requests
from dotenv import load_dotenv
import time
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go


# Loading environment variables from .env file
# This is necessary to use the TMDB API key
load_dotenv()  
TMDB_BEARER = os.getenv('TMDB_BEARER')
assert TMDB_BEARER, "Erro: defina TMDB_BEARER no seu .env com o token de leitura (v4)"


## 2. 🛠️ API Request Configuration and Data Retrieval

This block configures API calls, handles pagination, and saves new data to disk:

1. **Request Throttling**  
   - Defines `delay_seconds = 1` to pause between requests and respect rate limits.

2. **Re-loading and Validating the Bearer Token**  
   - Calls `load_dotenv()` again to ensure environment variables are loaded.  
   - Reads `TMDB_BEARER` and asserts its presence, raising:
     ```python
     Error: set TMDB_BEARER in your .env file with the v4 token
     ```

3. **Authorization Headers**  
   - Builds `headers` dict with:
     - `Authorization`: `Bearer {TMDB_BEARER}`
     - `Content-Type`: `application/json;charset=utf-8`

4. **Fetch Function Definition**  
   - `fetch_page(page=1)` sends a GET to `https://api.themoviedb.org/3/movie/popular`  
   - Passes `language='en-US'` and the `page` parameter.  
   - Raises an exception on HTTP errors and returns the parsed JSON.

5. **Determine Total Pages**  
   - Calls `fetch_page(1)` to read `total_pages`.  
   - Prints out:
     ```
     🔎 Total pages available: {total_pages}
     ```

6. **Limit Pages to Fetch**  
   - Sets `max_pages = 50` (adjustable).  
   - Computes `pages_to_fetch = min(total_pages, max_pages)` and prints:
     ```
     📥 Will fetch pages 1 through {pages_to_fetch}
     ```

7. **Load Existing Movie IDs**  
   - Ensures `data/raw/` exists.  
   - Iterates over any `popular_page_*.json` files, loading JSON and collecting existing movie IDs into `existing_ids`.  
   - Prints:
     ```
     🎬 {len(existing_ids)} movies already on disk
     ```

8. **Download Loop with Deduplication**  
   - Iterates pages `1` to `pages_to_fetch`:  
     - Fetches page data, sleeps for `delay_seconds`.  
     - Filters out movies whose IDs are already in `existing_ids`.  
     - If no new movies, prints:
       ```
       ⚠ Page {p}: no new movies, skipping
       ```
     - Otherwise:
       - Saves the new chunk to `data/raw/popular_page_{p}.json`.  
       - Extends `movies_raw` and updates `existing_ids`.  
       - Prints:
         ```
         ✅ Page {p}: fetched {len(new_chunk)} new movies
         ```

9. **Summary of Fetched Data**  
   - After the loop, prints:
     ```
     🎉 Total new movies fetched: {len(movies_raw)}
     ```


In [2]:

# Delay between API requests (in seconds)
delay_seconds = 1  # adjust as needed

# Load the Bearer Token (v4)
load_dotenv()
TMDB_BEARER = os.getenv('TMDB_BEARER')
assert TMDB_BEARER, "Error: set TMDB_BEARER in your .env file with the v4 token"

headers = {
    'Authorization': f'Bearer {TMDB_BEARER}',
    'Content-Type': 'application/json;charset=utf-8'
}

def fetch_page(page=1):
    url = "https://api.themoviedb.org/3/movie/popular"
    params = {'language': 'en-US', 'page': page}
    r = requests.get(url, headers=headers, params=params)
    r.raise_for_status()
    return r.json()

# 1) Discover how many total pages are available
first = fetch_page(1)
total_pages = first['total_pages']
print(f"🔎 Total pages available: {total_pages}")

# 2) Set how many pages to fetch (limit)
max_pages = 50  # adjust your limit
pages_to_fetch = min(total_pages, max_pages)
print(f"📥 Will fetch pages 1 through {pages_to_fetch}")

# 3) Load already existing movie IDs from data/raw/
raw_folder = Path("data/raw")
raw_folder.mkdir(parents=True, exist_ok=True)
existing_ids = set()
for f in raw_folder.glob("popular_page_*.json"):
    with open(f, encoding="utf-8") as fh:
        data = json.load(fh)
        existing_ids.update([m['id'] for m in data])
print(f"🎬 {len(existing_ids)} movies already on disk")

# 4) Download loop with delay and avoiding duplicates
movies_raw = []
for p in range(1, pages_to_fetch + 1):
    page_data = fetch_page(p)
    time.sleep(delay_seconds)
    new_chunk = [m for m in page_data['results'] if m['id'] not in existing_ids]
    if not new_chunk:
        print(f"⚠ Page {p}: no new movies, skipping")
        continue
    # Save raw JSON
    with open(raw_folder / f"popular_page_{p}.json", "w", encoding="utf-8") as f:
        json.dump(new_chunk, f, ensure_ascii=False, indent=2)
    movies_raw.extend(new_chunk)
    existing_ids.update([m['id'] for m in new_chunk])
    print(f"✅ Page {p}: fetched {len(new_chunk)} new movies")

print(f"🎉 Total new movies fetched: {len(movies_raw)}")



🔎 Total pages available: 50350
📥 Will fetch pages 1 through 50
🎬 553 movies already on disk
⚠ Page 1: no new movies, skipping
⚠ Page 2: no new movies, skipping
✅ Page 3: fetched 2 new movies
✅ Page 4: fetched 16 new movies
✅ Page 5: fetched 19 new movies
✅ Page 6: fetched 13 new movies
✅ Page 7: fetched 5 new movies
✅ Page 8: fetched 2 new movies
✅ Page 9: fetched 13 new movies
✅ Page 10: fetched 13 new movies
✅ Page 11: fetched 8 new movies
✅ Page 12: fetched 9 new movies
✅ Page 13: fetched 8 new movies
✅ Page 14: fetched 11 new movies
✅ Page 15: fetched 8 new movies
✅ Page 16: fetched 12 new movies
✅ Page 17: fetched 4 new movies
✅ Page 18: fetched 6 new movies
✅ Page 19: fetched 11 new movies
✅ Page 20: fetched 15 new movies
✅ Page 21: fetched 15 new movies
✅ Page 22: fetched 8 new movies
✅ Page 23: fetched 7 new movies
✅ Page 24: fetched 4 new movies
✅ Page 25: fetched 11 new movies
✅ Page 26: fetched 15 new movies
✅ Page 27: fetched 5 new movies
✅ Page 28: fetched 11 new movies
✅ 

## 3. 🗃️ Raw Data Loading and Cleaning

This section reads your saved JSON files, constructs a DataFrame, and prepares the data for analysis:

1. **Load Raw JSON Files**  
   - Iterates over all `data/raw/popular_page_*.json` files.  
   - Extends the `all_movies` list with each file’s contents.

2. **Create DataFrame & Select Columns**  
   - Converts `all_movies` into a `pd.DataFrame`.  
   - Keeps only `['id', 'title', 'release_date', 'popularity', 'vote_average', 'genre_ids']`.  
   - Ensures `genre_ids` is a tuple (for consistency).  
   - Drops duplicate rows by `id`.

3. **Parse and Clean Dates**  
   - Parses `release_date` to `datetime`, coercing errors to `NaT`.  
   - Drops any rows where `release_date` is missing.

4. **Fetch and Map Genres**  
   - Requests the genre list from TMDB (`/genre/movie/list`).  
   - Builds `genre_map` from ID → name.  
   - Applies `map_genres(ids)` to translate `genre_ids` into a list of genre names.

5. **Explode Genres for Analysis**  
   - Uses `df.explode('genres')` to create one row per genre per movie, producing `df_exploded`.

6. **Preview Clean DataFrame**  
   - Displays the first few rows with `df.head()` to confirm successful cleaning and mapping.


In [3]:
# Load raw JSON data from files
raw_folder = Path("data/raw")
all_movies = []
for file in raw_folder.glob("popular_page_*.json"):
    with open(file, encoding='utf-8') as f:
        all_movies.extend(json.load(f))

# Convert to DataFrame
df = pd.DataFrame(all_movies)

# Keep relevant columns and drop duplicates (by unique ID)
# Convert genre_ids list to tuple for consistency
if df['genre_ids'].apply(lambda x: isinstance(x, list)).any():
    df['genre_ids'] = df['genre_ids'].apply(lambda ids: tuple(ids))
df = df[['id', 'title', 'release_date', 'popularity', 'vote_average', 'genre_ids']]
# Drop duplicates based on 'id'
df = df.drop_duplicates(subset=['id'])

# Parse 'release_date' to datetime and drop missing to datetime and drop missing
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df = df.dropna(subset=['release_date'])

# Fetch genre map
genre_resp = requests.get(
    "https://api.themoviedb.org/3/genre/movie/list",
    headers=headers,
    params={'language': 'en-US'}
).json()
genre_map = {g['id']: g['name'] for g in genre_resp['genres']}

# Map genre IDs to names

def map_genres(ids):
    return [genre_map.get(i, "Unknown") for i in ids]

df['genres'] = df['genre_ids'].apply(map_genres)

# Explode 'genres' for per-genre analysis
df_exploded = df.explode('genres')

# Preview cleaned DataFrame
df.head()

,id,title,release_date,popularity,vote_average,genre_ids,genres
0,1241436,Warfare,2025-04-09,485.3223,7.158,"(10752, 28)","[War, Action]"
1,950387,A Minecraft Movie,2025-03-31,409.3111,6.300,"(10751, 35, 12, 14)","[Family, Comedy, Adventure, Fantasy]"
2,977294,Tin Soldier,2025-05-22,383.4020,0.000,"(28, 53)","[Action, Thriller]"
3,1144430,Last Bullet,2025-05-06,377.9668,6.821,"(28, 80, 53)","[Action, Crime, Thriller]"
4,447273,Snow White,2025-03-12,313.0346,4.365,"(10751, 14)","[Family, Fantasy]"


## 📊 4. Interactive & Holistic Exploratory Data Analysis

> In this section, I combine rigorous statistical methods, interactive visualizations, and narrative-driven storytelling to extract actionable insights from 60+ TMDB movies.

- **Multi‑Modal Visualization**: Histograms, boxplots, time‑series, and bar charts—fully interactive with Plotly for real‑time exploration.  
- **Clean Data Pipeline**: End‑to‑end flow from raw API JSON to pandas cleaning, date parsing, dynamic genre mapping, and artifact versioning.  
- **Stakeholder‑Centric**: Clear titles, labeled axes, and “why this matters” annotations that translate data patterns into business impact.  
- **User Empowerment**: Integrated `ipywidgets` slider & dropdown for on‑the‑fly filtering by year and genre—ideal for non‑technical audiences.  
- **Reproducibility & Scalability**: Modular functions and Git‑ignored `data/raw` & `data/processed` directories ensure audit‑friendly, production‑grade workflows.

> *Showcases technical fluency, data storytelling acumen, and best practices for end‑to‑end analytics.*  


### 4.1 ⭐ Rating Distribution  
**🔍 Insightful Distribution Analysis**  
- Combines histogram and boxplot to reveal central tendencies, dispersion, and outliers in movie ratings  
- `nbins=20` balances granularity vs. readability for spotting rating clusters  
- Interactive Plotly visuals allow stakeholders to hover for exact counts and quartiles  


In [4]:


# Set the renderer for classic Jupyter/Lab
pio.renderers.default = 'notebook_connected'

# (Optional) Load your DataFrame before this block:
# df = pd.read_csv('your_file.csv')
# Make sure df['vote_average'] already exists

# Histogram of movie ratings
fig = px.histogram(
    df,
    x='vote_average',
    nbins=20,
    title='Distribution of Movie Ratings'
)
fig.show()

# Boxplot of movie ratings
fig2 = px.box(
    df,
    y='vote_average',
    title='Boxplot of Movie Ratings'
)
fig2.show()


### 4.2 ⏳ Release Trends by Year  
**📈 Temporal Trend Detection**  
- Converts `release_date` to a numeric `year` field for time‑series grouping  
- Line chart highlights production spikes/dips, enabling correlation with industry events  
- Clear axis labels and titles make it presentation‑ready for executive meetings  


In [5]:


# Set the renderer for classic Jupyter/Lab
pio.renderers.default = 'notebook_connected'

# Ensure 'release_date' is datetime and extract year
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['year'] = df['release_date'].dt.year

# Group by year and count number of movies
time_series = (
    df
    .groupby('year')
    .size()
    .reset_index(name='count')
)

# Line chart of movies released per year
fig = px.line(
    time_series,
    x='year',
    y='count',
    title='Number of Movies Released per Year'
)
fig.update_layout(
    xaxis_title='Year',
    yaxis_title='Number of Movies'
)
fig.show()


### 4.3 🎯 Genre Popularity  
**📊 Genre Demand Profiling**  
- Explodes multi‑genre lists to capture each film’s full genre footprint  
- Bar chart with categorical x‑axis enforces logical ordering for comparison  
- Identifies high‑value genres to inform content strategy, recommendation engines, and marketing  


In [6]:
# 4.3 🎥 Genre Popularity (fixed for list-of-strings genres)
import pandas as pd
import plotly.express as px
import plotly.io as pio

# 1) Set renderer for Jupyter/Lab
pio.renderers.default = 'notebook_connected'

# 2) Explode the list-of-genres directly
df_exploded = (
    df
    .dropna(subset=['genres'])   # drop any row where genres is null
    .explode('genres')            # directly explode the Python list
    .reset_index(drop=True)
)

# Quick sanity check
print("Unique genres after explode:", df_exploded['genres'].unique()[:10])

# 3) Build counts DataFrame
genre_counts = (
    df_exploded['genres']
      .value_counts()            # count each genre
      .rename_axis('genre')      # name the index axis
      .reset_index(name='count') # turn index → column 'genre', values → 'count'
)

print("Top genres:\n", genre_counts.head(5))

# 4) Plot bar chart
fig = px.bar(
    genre_counts,
    x='genre',
    y='count',
    title='Movies per Genre'
)
fig.update_layout(
    xaxis_title='Genre',
    yaxis_title='Number of Movies',
    xaxis=dict(
        type='category',
        categoryorder='total descending',
        categoryarray=genre_counts['genre']
    )
)
fig.show()





Unique genres after explode: ['War' 'Action' 'Family' 'Comedy' 'Adventure' 'Fantasy' 'Thriller' 'Crime'
 'Science Fiction' 'Drama']
Top genres:
        genre  count
0     Action    199
1      Drama    160
2  Adventure    149
3   Thriller    145
4     Comedy    121


## 5. 🔄 Interactive Popularity vs. Rating Scatter with Widgets

This cell brings together data prep, Plotly rendering, and ipywidgets controls to create a dynamic scatter plot:

1. **Plotly Notebook Renderer**  
   - Use `pio.renderers.default = 'notebook'` so charts display inline in Jupyter.

2. **Ensure `year` Column Is Integer**  
   - If `year` already exists, cast it to `int`.  
   - Otherwise, parse `release_date` to datetime and extract the year.

3. **Normalize & Explode Genres**  
   - Convert the `genres` string (e.g. `"Action|Comedy|Drama"`) into a list via `str.split('|')`.  
   - Use `.explode('genres')` so each row corresponds to a single genre, resetting the index for consistency.

4. **Configure Widgets**  
   - **Year Slider** (`IntRangeSlider`): spans from the earliest to latest year in your data.  
   - **Genre Dropdown**: options include “All” plus every unique genre found.

5. **Update Function**  
   - Filters `df_exploded` by the selected year range and genre (if not “All”).  
   - If no data remains after


In [ ]:


# Renderer do Plotly no notebook
pio.renderers.default = 'notebook'

# --- 1) Garante coluna 'year' como inteiro ---
if 'year' not in df.columns:
    df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
    df['year'] = df['release_date'].dt.year
else:
    df['year'] = df['year'].astype(int)

# --- 2) Explode genres para df_exploded ---
# Supondo que df['genres'] seja uma string tipo "Action|Comedy|Drama"
df_exploded = (
    df
    .assign(genres=df['genres'].str.split('|'))
    .explode('genres')
    .reset_index(drop=True)
)

# --- 3) Widgets de controle ---
min_year, max_year = int(df['year'].min()), int(df['year'].max())
year_slider = widgets.IntRangeSlider(
    value=[min_year, max_year],
    min=min_year, max=max_year,
    step=1, description='Year Range:'
)

genre_options = ['All'] + sorted(df_exploded['genres'].dropna().unique().tolist())
genre_dropdown = widgets.Dropdown(
    options=genre_options, value='All', description='Genre:'
)

# --- 4) Função de atualização ---
def update_plots(year_range, genre):
    start, end = year_range
    filtered = df_exploded[
        (df_exploded['year'] >= start) &
        (df_exploded['year'] <= end)
    ]
    if genre != 'All':
        filtered = filtered[filtered['genres'] == genre]
    # Se não houver dados, exibe mensagem
    if filtered.empty:
        print("Nenhum filme encontrado para esses filtros.")
        return

    scatter = px.scatter(
        filtered,
        x='popularity',
        y='vote_average',
        hover_data=['title', 'release_date'],
        title=f'Popularity vs Rating ({genre}), {start}–{end}'
    )
    scatter.show()

# --- 5) Exibição ---
ui = widgets.interactive_output(
    update_plots,
    {'year_range': year_slider, 'genre': genre_dropdown}
)
display(widgets.VBox([year_slider, genre_dropdown]), ui)


Output()

## 6️⃣ Compute & Visualize Key Metrics

This code cell performs four core analyses and renders interactive visualizations:

1. **Mid‑range Ratings (6–8)**  
   - Calculates the percentage of movies with ratings between 6.0 and 8.0.

2. **Pandemic Impact (2019 → 2020)**  
   - Compares release counts in 2019 vs. 2020, computing the percentage drop.

3. **Popularity vs. Rating Correlation**  
   - Measures the relationship between “hype” (popularity) and quality (vote_average).

4. **Genre Demand Profiling**  
   - Groups by genre to compute title counts and average ratings, sorted by volume.

---

### 🚀 Textual Output
- `Mid‑range ratings (6–8): X.X%`  
- `High-rating outliers (>9): [...]`  
- `Low-rating outliers (<5): [...]`  
- `2020 releases dropped Y.Y% vs. 2019`  
- Top 5 genres by count and their average ratings  
- `Correlation(popularity, rating) = Z.ZZ`

---

### 📊 Visualizations
- **Pie Chart**: Proportion of films rated 6–8 vs. others  
- **Bar Chart**: Releases in 2019 vs. 2020 (with % drop)  
- **Scatter Plot**: Popularity vs. Rating (hover shows movie title)  

> **Why recruiters will love this:**  
> - Numbered structure and self-explanatory steps  
> - Clear, concise comments for each block  
> - Descriptive chart titles for immediate context  
> - Strategic use of emojis for readability  
> - Modular code ready for seamless notebook integration  


In [ ]:

# Ensure required variables are computed in this cell
# 1) Mid-range rating percentage
pct_mid = ((df['vote_average'] >= 6) & (df['vote_average'] <= 8)).mean() * 100

# 2) Pandemic impact on releases
year_counts = df['year'].value_counts().sort_index()
pct_drop = (year_counts.loc[2020] / year_counts.loc[2019] - 1) * 100

# 3) Popularity vs. rating correlation
corr = df[['popularity', 'vote_average']].corr().iloc[0,1]

# 4) Genre demand profiling
genre_stats = (
    df_exploded.groupby('genres')
    .agg(count=('id','size'), avg_rating=('vote_average','mean'))
    .sort_values('count', ascending=False)
)

# Print textual insights
print(f"🟢 Mid-range ratings (6–8): {pct_mid:.1f}%")
high_outliers = df[df['vote_average'] > 9]['title'].tolist()
low_outliers = df[df['vote_average'] < 5]['title'].tolist()
print(f"🌟 High-rating outliers (>9): {high_outliers}")
print(f"⚠️ Low-rating outliers (<5): {low_outliers}")
print(f"📉 2020 releases dropped {pct_drop:.1f}% vs. 2019")
print(genre_stats.head(5))
print(f"🔗 Corr(popularity, rating) = {corr:.2f}")

# --- Plots ---
# Pie chart for mid-range
fig1 = go.Figure(go.Pie(
    labels=['6–8','Others'],
    values=[pct_mid, 100 - pct_mid],
    hole=0.4,
    title='Percentage of Movies Rated 6–8'
))
fig1.show(renderer='notebook_connected')

# Bar chart for 2019 vs 2020 releases
fig2 = go.Figure(go.Bar(
    x=[2019, 2020],
    y=[year_counts.loc[2019], year_counts.loc[2020]],
    text=[year_counts.loc[2019], year_counts.loc[2020]],
    textposition='auto'
))
fig2.update_layout(
    title=f'Releases: 2019 vs 2020 (Drop {pct_drop:.1f}%)',
    xaxis_title='Year', yaxis_title='Number of Releases'
)
fig2.show(renderer='notebook_connected')

# Scatter for popularity vs rating
fig3 = go.Figure(go.Scatter(
    x=df['popularity'], y=df['vote_average'], mode='markers',
    marker=dict(size=6, opacity=0.6), text=df['title'],
    hovertemplate='<b>%{text}</b><br>Popularity: %{x}<br>Rating: %{y}<extra></extra>'
))
fig3.update_layout(
    title=f'Popularity vs Rating (corr={corr:.2f})',
    xaxis_title='Popularity', yaxis_title='Vote Average'
)
fig3.show(renderer='notebook_connected')


🟢 Mid-range ratings (6–8): 73.5%
🌟 High-rating outliers (>9): ['Salome', 'Flame of Recca: Final Burning']
⚠️ Low-rating outliers (<5): ['Tin Soldier', 'Snow White', 'Conjuring the Cult', 'Easter Bloody Easter', 'El Cas Àngelus. La fascinació de Dalí', 'Swarm', 'From the World of John Wick: Ballerina', 'Ligaw', 'The Old Guard 2', 'F1 The Movie', 'Nobody 2', 'Lilo & Stitch', 'The Jowa Collector', 'Demon Slayer: Kimetsu no Yaiba Infinity Castle', 'Operation Undead', 'Fantasm Comes Again', 'Red', 'Materialists', 'Wife Swap', 'The Conjuring: Last Rites', 'Breathing In', 'Betray: Thirst', 'EFC']
📉 2020 releases dropped -57.1% vs. 2019
Empty DataFrame
Columns: [count, avg_rating]
Index: []
🔗 Corr(popularity, rating) = -0.11


## 🚀 Step 7: Enrich Dataset with Financials

In this cell, we augment our cleaned movie DataFrame by fetching **revenue** and **budget** details for each film, then persist the enriched data for downstream analysis:

1. **Prepare the DataFrame**  
   - Create a copy of `df` (from earlier cleaning) to avoid mutating the original.

2. **Secure Authentication**  
   - Reload environment variables and pull `TMDB_BEARER` from `.env`  
   - Build an `Authorization: Bearer …` header for all subsequent TMDB requests.

3. **Define `fetch_financials()`**  
   - Hits the `/movie/{movie_id}` endpoint with our bearer token  
   - Returns a `(revenue, budget)` tuple by parsing the JSON response

4. **Row‑Wise Enrichment**  
   - Iterate over `df_clean.iterrows()`  
   - For each movie ID, call `fetch_financials()` and assign the returned values into new `revenue` and `budget` columns  
   - This approach ensures every film carries its financial context

5. **Persist Enriched Data**  
   - Guarantee the `data/processed/` directory exists via `pathlib`  
   - Save the full enriched DataFrame to `movies_with_financials.csv` (no index)  
   - Print a completion message to confirm success

> **Outcome:**  
> You now have a single CSV combining metadata, ratings, genres **and** financials—ready for advanced EDA, modeling, or dashboarding.


In [ ]:

# Reuse DataFrame from earlier steps
df_clean = df.copy()

# Load TMDB Bearer token
load_dotenv()
TMDB_BEARER = os.getenv('TMDB_BEARER')
headers = {'Authorization': f'Bearer {TMDB_BEARER}'}

# Function to fetch revenue and budget
def fetch_financials(mid):
    url = f"https://api.themoviedb.org/3/movie/{mid}"
    resp = requests.get(url, headers=headers, params={'language': 'en-US'})
    resp.raise_for_status()
    data = resp.json()
    return data.get('revenue'), data.get('budget')

# Enrich each row
for i, row in df_clean.iterrows():
    rev, bud = fetch_financials(row['id'])
    df_clean.at[i, 'revenue'] = rev
    df_clean.at[i, 'budget'] = bud

# Ensure processed folder exists
Path("data/processed").mkdir(parents=True, exist_ok=True)

# Save enriched data
df_clean.to_csv('data/processed/movies_with_financials.csv', index=False)
print('✅ Data enrichment complete!')

✅ Data enrichment complete!


## 🎯 Step 8: Summary Metrics by Genre

In this cell, we take our fully processed movie dataset (which includes financial information) and compute key summary statistics for each genre:

1. **Load the Data**  
   - Read in the cleaned CSV (`movies_with_financials.csv`) and parse the `release_date` column as datetime.

2. **Normalize the Genre List**  
   - Use `explode('genres')` to turn each movie’s genre list into individual rows, so that each genre can be analyzed separately.

3. **Group & Aggregate**  
   - **Average Rating** (`vote_average`): mean score across all movies in the genre  
   - **Movie Count** (`id`): number of movies tagged with the genre  
   - **Average Popularity** (`popularity`): mean popularity index  
   - **Total Revenue** (`revenue`): sum of box office revenue  
   - **Total Budget** (`budget`): sum of production budgets  

4. **Sort & Inspect**  
   - Sort genres by descending movie count to highlight the most prolific categories.  
   - Display the top 10 genres for a quick, at‑a‑glance summary.

> **Insight:** This overview lets us see which genres dominate in volume, budget, and revenue, and how their average ratings and popularity scores compare.



In [ ]:
# Compute summary metrics by genre
from ast import literal_eval
df_fin = pd.read_csv('data/processed/movies_with_financials.csv', parse_dates=['release_date'])

# CSV stores the genre lists as text; decode before exploding.
def parse_genres(value):
    if isinstance(value, list):
        parsed = value
    elif pd.isna(value):
        return []
    else:
        parsed = literal_eval(value)
    if not isinstance(parsed, list) or any(not isinstance(item, str) for item in parsed):
        raise ValueError("Expected a list of genre names")
    return parsed

df_fin['genres'] = df_fin['genres'].map(parse_genres)

df_seg = (
    df_fin.explode('genres')
    .groupby('genres')
    .agg(
        avg_rating=('vote_average','mean'),
        movie_count=('id','count'),
        avg_popularity=('popularity','mean'),
        total_revenue=('revenue','sum'),
        total_budget=('budget','sum')
    )
    .sort_values('movie_count', ascending=False)
)

df_seg.head(10)

## 9. 🔢 Predictive Modeling: Linear Regression for Movie Ratings

In this cell, we build a simple baseline model to estimate a movie’s average rating from its popularity and budget:

1. **Import Modeling Tools**  
   - `train_test_split` to create hold‑out test data  
   - `LinearRegression` for a basic predictive framework  
   - `mean_squared_error` to quantify prediction error

2. **Load & Prepare Features**  
   - Read the enriched CSV (`movies_with_financials.csv`) into `df_mod`  
   - Define **X** as the two numerical drivers:  
     - `popularity` (how much buzz the movie generates)  
     - `budget` (production spend)  
     - Fill any missing financials with zero  
   - Define **y** as the target: `vote_average`

3. **Train/Test Split**  
   - Reserve 20% of the data for testing (`test_size=0.2`)  
   - Set `random_state=42` for reproducible splits

4. **Fit & Predict**  
   - Train the linear model on the training fold  
   - Generate predictions on the held‑out test set

5. **Evaluate Performance**  
   - Compute and print the **Test MSE** (mean squared error) to gauge how well budget and popularity explain rating variation


In [ ]:
# Linear regression to predict movie ratings
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error

# Load enriched data
df_mod = pd.read_csv('data/processed/movies_with_financials.csv')
X = df_mod[['popularity','budget']].fillna(0)
y = df_mod['vote_average']

# Train/Test split and modeling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)
preds = model.predict(X_test)
print('Test MSE:', mean_squared_error(y_test, preds))
# Compare against a training-mean baseline on the same split.
baseline = DummyRegressor(strategy='mean').fit(X_train, y_train)
baseline_mse = mean_squared_error(y_test, baseline.predict(X_test))
model_mse = mean_squared_error(y_test, preds)
print('Training-mean baseline Test MSE:', baseline_mse)
print('MSE improvement over baseline:', baseline_mse - model_mse)
